In [6]:
import os
import re
import cv2
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Define your exact project paths
ROOT = Path("C:\\Users\\rik_y\\Documents\\GitHub\\ricardoserrano.github.io\\Carcassone_Companion_App\\tile_library_z-man_2014")
IMAGE_SOURCE = ROOT / "crops_training_set"
MASK_SOURCE = ROOT / "Segmentation_mask_09132026/SegmentationClass"

class CarcassonneSplitDataset(Dataset):
    def __init__(self, file_list, images_dir, masks_dir, target_size=(256, 256)):
        """
        Accepts an explicit list of filenames to construct the split.
        """
        self.valid_images = file_list
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.target_size = target_size
        
        # OpenCV reads BGR. This mapping is flipped to (Blue, Green, Red)
        self.bgr_to_id = {
            (0, 0, 0): 0,       # Background / background
            (80, 80, 178): 1,   # City
            (255, 221, 51): 2,  # Farm
            (61, 245, 61): 3,   # Field
            (103, 157, 250): 4, # Garden
            (83, 50, 250): 5,   # Monastery
            (209, 240, 170): 6, # Road
            (183, 50, 250): 7,  # RoadBlock
            (23, 102, 169): 8   # Shield
        }

    def _bgr_to_class_mask(self, mask_bgr):
        height, width, _ = mask_bgr.shape
        class_mask = np.zeros((height, width), dtype=np.int64)
        for bgr, class_id in self.bgr_to_id.items():
            match_condition = (mask_bgr[:, :, 0] == bgr) & \
                              (mask_bgr[:, :, 1] == bgr) & \
                              (mask_bgr[:, :, 2] == bgr)
            class_mask[match_condition] = class_id
        return class_mask

    def __len__(self):
        return len(self.valid_images)

    def __getitem__(self, idx):
        img_name = self.valid_images[idx]
        img_path = self.images_dir / img_name
        mask_name = img_path.stem + ".png"
        mask_path = self.masks_dir / mask_name
        
        image = cv2.imread(str(img_path))
        mask = cv2.imread(str(mask_path))
        
        image = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)
        
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        class_mask = self._bgr_to_class_mask(mask)
        
        image_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(class_mask).long()
        
        return image_tensor, mask_tensor

# --- Step 1: Discover all matching files first ---
filename_regex = re.compile(r'^IMG_\d+\.(jpg|jpeg)$', re.IGNORECASE)
all_images = sorted([
    f.name for f in IMAGE_SOURCE.iterdir() 
    if f.is_file() and filename_regex.match(f.name)
])

# --- Step 2: Split filenames (80% Train, 20% Validation) ---
# random_state keeps the split reproducible across script runs
train_files, val_files = train_test_split(all_images, test_size=0.2, random_state=42)

print(f"Total matching images found: {len(all_images)}")
print(f"Allocated {len(train_files)} to training and {len(val_files)} to validation.")

# --- Step 3: Instantiate Dataset objects ---
train_dataset = CarcassonneSplitDataset(train_files, IMAGE_SOURCE, MASK_SOURCE)
val_dataset = CarcassonneSplitDataset(val_files, IMAGE_SOURCE, MASK_SOURCE)

# --- Step 4: Create DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)


Total matching images found: 16
Allocated 12 to training and 4 to validation.


In [7]:
# Highly recommended for Carcassonne: give higher weight to rare elements like Shields and RoadBlocks
# These weights tell the loss function to care more about small features.
# Layout: [Background, City, Farm, Field, Garden, Monastery, Road, RoadBlock, Shield]
# class_weights = torch.tensor([1.0, 2.0, 1.5, 1.2, 3.0, 4.0, 2.0, 5.0, 5.0]).cuda() 

# Note: Remove .cuda() if you are training purely on a laptop CPU:
class_weights = torch.tensor([1.0, 2.0, 1.5, 1.2, 3.0, 4.0, 2.0, 5.0, 5.0])

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)


In [8]:
import torch
import torch.optim as optim
from transformers import SegformerForSemanticSegmentation

# 1. Device Configuration (Leverage lightweight GPU acceleration if your laptop has it)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training will run on: {device}")

# 2. Model Initialization (SegFormer-B0 is ideal for laptops)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=9, # Your 9 distinct Carcassonne classes
    id2label={0: "Background", 1: "City", 2: "Farm", 3: "Field", 4: "Garden", 5: "Monastery", 6: "Road", 7: "RoadBlock", 8: "Shield"},
    label2id={"Background": 0, "City": 1, "Farm": 2, "Field": 3, "Garden": 4, "Monastery": 5, "Road": 6, "RoadBlock": 7, "Shield": 8},
    ignore_mismatched_sizes=True # Safely overrides default ImageNet head
)
model.to(device)

# 3. Optimizer & Loss Setup
# AdamW works best for Transformers; a learning rate of 6e-5 prevents explosions on tiny batch sizes
optimizer = optim.AdamW(model.parameters(), lr=6e-5, weight_decay=0.01)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))

# 4. Training Control Parameters
num_epochs = 15  # Start small to test laptop thermals and performance
best_val_loss = float('inf')

# 5. Loop Execution
for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(pixel_values=images)
        logits = outputs.logits  # Shape: (Batch, Num_Classes, H/4, W/4)
        
        # SegFormer outputs maps downsampled by 4; interpolate back up to match your target mask size (256x256)
        upsampled_logits = torch.nn.functional.interpolate(
            logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
        )
        
        loss = criterion(upsampled_logits, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    epoch_train_loss = train_loss / len(train_loader.dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(pixel_values=images)
            logits = outputs.logits
            
            upsampled_logits = torch.nn.functional.interpolate(
                logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
            )
            
            loss = criterion(upsampled_logits, masks)
            val_loss += loss.item() * images.size(0)
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    # Save checkpoint if validation improves
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_carcassonne_model.pth")
        print("💾 Best model weights saved!")

print("Training cycle complete.")


[transformers] You passed `num_labels=9` which is incompatible to the `id2label` map of length `1000`.


Training will run on: cpu


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.classifier.bias                             

ValueError: operands could not be broadcast together with shapes (256,256) (3,) 